In [1]:
# epoch影响较大
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/database/PDAC/PDAC_SC_filtered.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/database/PDAC/PDAC_ST.h5ad" \
    --n_epochs 300 \
    --resolution 3 \
    --loss_type zinb \
    --beta 0.1 \
    --lambda_mmd 1.0 \
    --top_n_per_type 200 \
    --latent_dim 256 \
    --output_dir ../deconv_results/PDAC \
    --aggregation_method mean
# --precomputed_marker_file "/mnt/d/ST_Graduation_Project/SC_MAP_ST/deconv_results/PDAC/final_genes.txt"

/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 200
   Leiden resolution: 3.0
   Batch size: 256
   Epochs: 300
   Learning rate: 0.001
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: ZINB
   Lambda MMD: 1.0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/database/PDAC/PDAC_SC_filtered.h5ad
   SC shape: (1804, 19738)
   SC avg counts/cell: 3345.6 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/database/PDAC/PDAC_ST.h5ad
   ST shape: (428, 19738)
   Common genes: 19738
   SC final: (1804, 19738)
   ST final: (428, 19738)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 32 clusters
Marker genes per cluster:
   0: 58 -> 58 (after Lasso)
   1: 200 -> 111 (after Lasso)
   10: 97 -> 97 (after Lasso)
   11: 46 -> 46 (after Lasso)
   12: 58 -> 50 (after Lasso)
   13: 26 -> 26 (aft

VAE Training:  40%|████      | 121/300 [00:40<00:59,  2.99epoch/s, Train=1498.9661, Recon=1493.0149, KL=59.5117, MMD=0.0171, Test=1596.3657]



⚠️ Early stopping triggered at epoch 122/300
   Best test loss: 1590.0280, Current test loss: 1596.3657
   No improvement for 20 consecutive epochs
📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (1804, 1076)
   Number of clusters: 32
   Computed embeddings shape: (1804, 256)
Computing cluster centers with mean aggregation...
      Cluster 0: 117 cells (mean aggregation)
      Cluster 1: 108 cells (mean aggregation)
      Cluster 2: 71 cells (mean aggregation)
      Cluster 3: 61 cells (mean aggregation)
      Cluster 4: 55 cells (mean aggregation)
      Cluster 5: 55 cells (mean aggregation)
      Cluster 6: 53 cells (mean aggregation)
      Cluster 7: 52 cells (mean aggregation)
      Cluster 8: 50 cells (mean aggregation)
      Cluster 9: 49 cells (mean aggregation)
      Cluster 10: 47 cells (mean aggregation)
      Cluster 11: 45 cells (mean aggregation)
      Cluster 12: 103 cells (mean aggregation)
      Cluster 13: 44 cells (mean aggregation)
  

In [3]:
# λ 越大更稀疏0.1-0.5
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/database/PDAC/PDAC_ST.h5ad" \
    --stage1_model_path "../deconv_results/PDAC/final_vae.pth" \
    --output_dir "../deconv_results/PDAC" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 5e-4 \
    --loss_lambda_mse 0.5 \
    --loss_lambda_cosine 5 \
    --loss_lambda_pearson 5 \
    --loss_lambda_reg 0.1 \
    --loss_lambda_sparse 0.1 \
    --loss_lambda_proportion 0.1 \
    --k_spatial 10 \
    --k_celltype 20 \
    --batch_size 512 \
    --n_epochs 400 \
    --weight_threshold 0.05

Sample name: PDAC
Stage 1 model: ../deconv_results/PDAC/final_vae.pth
Output directory: ../deconv_results/PDAC
Device: cpu
Weight threshold: 0.05
Loading pretrained VAE Encoder...
   VAE architecture: 1076 -> 256
   Output type: zinb
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 1804 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/PDAC/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([32, 256])
   Cluster expressions (marker): torch.Size([32, 1076])
   Cluster expressions (all genes): 32 × 19738
   Loaded celltype mapping: 32 clusters
   Average cell counts: 3345.6
Loaded all genes list: 19738 genes
VAE Encoder loaded: 1076 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '4', '5', '6', '7', '8', '9']
Marker genes: 1076
Using Stage 1 cluster centers and expressions...
Loaded 32 clusters
Using Stage 1

GAT Training:  66%|██████▌   | 263/400 [03:42<01:55,  1.18epoch/s, Total=12.1790, Pearson=0.5818, MSE=13.8442, Cosine=0.4543, P_pat=10, M_pat=14, C_pat=10]



⚠️ Early stopping triggered at epoch 264/400
   All three core losses stopped improving:
      Pearson: best=0.5815, current=0.5818, no improvement for 10 epochs
      MSE: best=13.5429, current=13.8442, no improvement for 14 epochs
      Cosine: best=0.4540, current=0.4543, no improvement for 10 epochs
Evaluating model results...
Applying weight threshold: 0.05
   Non-zero elements: 8560 -> 2245 (16.4%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/PDAC/PDAC_reconstructed_all_genes.csv
   Cell type composition...
   Found duplicate celltype names: ['Ductal', 'Cancer']. Merging corresponding cluster columns by summing weights.
   Columns before: 32, after merge: 7
   Saved cell composition (celltype): ../deconv_results/PDAC/PDAC_cell_composition.csv
   Saved cluster composition: ../deconv_results/PDAC/PDAC_cluster_composition.csv
   Using 1076 marker genes for reconstruction quality 